# Clase 012 — Logging

**Parte 0** · Logging HOWTO.

> 🎯 Dejar `print` para debug y usar `logging` con niveles, handlers, formatters. La diferencia entre código observable y código que adivinas.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import logging
import tempfile
from pathlib import Path
from logging.config import dictConfig

## 1️⃣ Por qué `logging` y no `print`

| `print` | `logging` |
|---|---|
| stdout fijo | múltiples destinos |
| sin nivel | DEBUG/INFO/WARNING/ERROR/CRITICAL |
| sin contexto | módulo, función, timestamp automáticos |
| no se silencia sin tocar código | filtras por nivel |
| no estructurado | parseable, integrable con observabilidad |

## 2️⃣ Niveles

| Nivel | Uso |
|---|---|
| `DEBUG` | detalles para diagnóstico (variables, flujo) |
| `INFO` | progreso normal ("cargados 1000 registros") |
| `WARNING` | algo raro pero no fatal ("valor por defecto usado") |
| `ERROR` | la operación falló ("no se pudo cargar el CSV") |
| `CRITICAL` | sistema no puede continuar |

Filtro: si configuras nivel `INFO`, solo se muestran INFO/WARNING/ERROR/CRITICAL.

## 3️⃣ Setup mínimo con `basicConfig`

```python
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)
log = logging.getLogger(__name__)
log.info('arrancando...')
log.warning('cuidado')
```

⚠️ **Gotcha**: `basicConfig` solo aplica si **no había handlers** en root. En Jupyter (kernel reusado) puede no tener efecto — usa `force=True`.

In [ ]:
# Reset y configuración explícita para notebook
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

log = logging.getLogger(__name__)
log.debug('detalle interno')
log.info('todo OK')
log.warning('algo raro')
log.error('fallo manejable')

## 4️⃣ Logger por módulo — la práctica correcta

```python
# loader.py
import logging
log = logging.getLogger(__name__)   # 'loader' o 'mi_pkg.loader'

def cargar(path):
    log.info(f'cargando {path}')
    ...
```

Ventaja: el root logger configurado **una vez** propaga a todos los módulos. Puedes silenciar uno solo con `logging.getLogger('loader').setLevel(WARNING)`.

In [ ]:
# Simula 2 módulos
log_app = logging.getLogger('mi_app')
log_db  = logging.getLogger('mi_app.db')

log_app.info('arrancando app')
log_db.info('conectando a db')
log_db.warning('latencia alta')

# Silencia un módulo específico
log_db.setLevel(logging.ERROR)
log_db.info('esto NO se ve')
log_db.error('esto sí se ve')

## 5️⃣ Handler doble — consola + archivo

Para producción típicamente queremos:
- Consola: INFO+ (lo que el operador ve)
- Archivo: DEBUG+ (todo para post-mortem)

In [ ]:
log_file = Path(tempfile.gettempdir()) / 'demo_app.log'

config = {
    'version': 1,
    'disable_existing_loggers': False,
    'formatters': {
        'verbose': {'format': '%(asctime)s [%(levelname)s] %(name)s: %(message)s'},
        'corto':   {'format': '[%(levelname)s] %(message)s'},
    },
    'handlers': {
        'consola': {
            'class': 'logging.StreamHandler',
            'level': 'INFO',
            'formatter': 'corto',
        },
        'archivo': {
            'class': 'logging.FileHandler',
            'filename': str(log_file),
            'level': 'DEBUG',
            'formatter': 'verbose',
            'mode': 'w',
        },
    },
    'root': {
        'level': 'DEBUG',
        'handlers': ['consola', 'archivo'],
    },
}
dictConfig(config)

log = logging.getLogger('demo')
log.debug('DEBUG: solo va al archivo')
log.info('INFO: va a ambos')
log.warning('WARN: va a ambos')
log.error('ERROR: va a ambos')

print(f'\n--- contenido de {log_file} ---')
print(log_file.read_text())

## 6️⃣ Buenas prácticas

- **No hagas `f'{var}'` en el mensaje** si vas a filtrar por nivel: pasa args separados, `log.debug('valor: %s', var)`. Así no se formatea si el nivel no aplica.
- **No loguees datos sensibles**: PII, tokens, contraseñas. Filtra antes.
- **`exc_info=True`** en `log.error` para capturar el traceback completo.
- **Logger por módulo**, configuración por aplicación. No mezcles.

In [ ]:
# exc_info=True para capturar traceback
try:
    1 / 0
except ZeroDivisionError:
    log.error('división falló', exc_info=True)

## ✅ Checklist

- [ ] Sé los 5 niveles y cuándo usar cada uno
- [ ] Uso `getLogger(__name__)` en cada módulo
- [ ] Configuro logging UNA vez en el entrypoint
- [ ] Tengo handler consola (INFO+) y archivo (DEBUG+)
- [ ] Uso `exc_info=True` para errores con stacktrace

## 📝 Homework

Ver `README.md`. Notebook + 2 módulos + `logging_config.py` con `dictConfig`; entrega `app.log`.

## 📖 Definiciones y características

**Logger**

Punto de entrada para emitir logs. Se obtiene con `logging.getLogger(__name__)` — esto crea un logger nombrado por el módulo. Característica clave: los loggers son **jerárquicos** (separados por `.`); la config de root propaga a hijos.

**Handler**

Define **a dónde** van los logs (consola, archivo, syslog, sentry…). Un logger puede tener N handlers. Cada handler tiene su propio nivel y formatter.

**Formatter**

Define **cómo** se renderiza el log: `'%(asctime)s [%(levelname)s] %(name)s: %(message)s'`. Campos comunes: `asctime`, `levelname`, `name` (logger), `message`, `funcName`, `lineno`, `pathname`.

**Nivel (DEBUG/INFO/WARNING/ERROR/CRITICAL)**

Severidad ascendente. Filtran qué se emite. **DEBUG**: detalle interno; **INFO**: progreso normal; **WARNING**: algo raro pero no fatal; **ERROR**: operación falló; **CRITICAL**: sistema no puede continuar.

**`dictConfig`**

Forma declarativa de configurar logging desde un dict (o YAML/JSON). Más mantenible que llamadas `basicConfig`/`addHandler` dispersas. Convención: una sola llamada en el entrypoint.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `logging.basicConfig(...)` no tiene efecto | `basicConfig` solo aplica si root no tiene handlers ya. **Fix**: `basicConfig(..., force=True)` o limpia handlers (`for h in logging.root.handlers[:]: logging.root.removeHandler(h)`). |
| Logs duplicados (cada mensaje aparece 2 veces) | Configuraste el mismo handler dos veces (común al recargar módulos en Jupyter). **Fix**: limpia handlers antes de añadir, o usa `dictConfig` con `disable_existing_loggers=False` cuidadosamente. |
| `log.debug(f'valor: {expensive_call()}')` siempre evalúa | El f-string se construye **antes** de pasar a `debug` — el filtro de nivel no ayuda. **Fix**: usa lazy: `log.debug('valor: %s', expensive_call)` (sin paréntesis ⇒ solo se llama si pasa el filtro). |
| Mi logger emite en INFO pero quiero ver DEBUG | El nivel está en handler o logger raíz. **Fix**: `logging.getLogger().setLevel(logging.DEBUG)` Y `handler.setLevel(logging.DEBUG)` (ambos deben permitirlo). |
| `logging` rompe en multiprocessing | Handlers no son fork-safe. **Fix**: en cada proceso, configura logging de nuevo; o usa `QueueHandler` + `QueueListener` del cookbook oficial. |

## ❓ Preguntas frecuentes

**❓ ¿`print` o `logging`?**

**`logging` siempre en código que vivirá >1 día.** `print` solo para REPL/scripts one-shot. Logging te da niveles, timestamps, módulo origen, múltiples destinos, integración con observabilidad.

**❓ ¿Dónde configuro logging?**

**Una sola vez** en el entrypoint (`__main__`, `app.py`, `cli.py`). Cada módulo solo hace `log = logging.getLogger(__name__)`; nunca llama a `basicConfig`/`addHandler` desde un módulo importable.

**❓ ¿Por qué `getLogger(__name__)`?**

Crea un logger jerárquico nombrado por el módulo. Permite silenciar uno específico (`logging.getLogger('mi_app.db').setLevel(WARNING)`) sin tocar el resto. Pattern estándar.

**❓ ¿Cómo formato un dict/object en el mensaje?**

`log.info('user=%s data=%s', user_id, data)` (mejor que f-string por el lazy). Para JSON estructurado, usa `python-json-logger` o stdlib con custom formatter.

**❓ ¿`logging` propaga al root logger?**

Por default, sí — cada logger propaga al padre hasta root. Si configuras handlers en root y en hijos, verás el mensaje dos veces. **Fix**: `logger.propagate = False` en el hijo, o solo configura root.

## 🔗 Referencias

- [Logging HOWTO](https://docs.python.org/3/howto/logging.html)
- [Logging Cookbook](https://docs.python.org/3/howto/logging-cookbook.html)

➡️ **Siguiente:** [013 — Type hints y mypy](../013-type-hints-y-mypy/README.md)